# 02 Cleaning & Enrichment

Clean the raw data according to project requirements, engineer new features, and export processed files to `data/processed/`.

In [25]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/raw/Bookings.csv')
print(f"Initial shape: {df.shape}")

Initial shape: (103024, 21)


## 1. Drop Junk Columns

In [ ]:
junk_cols = ['Vehicle Images', 'Customer_ID']
cols_to_drop = [col for col in junk_cols if col in df.columns]

# Also check for fully null unnamed columns dynamically
unnamed_null_cols = [col for col in df.columns if 'Unnamed' in str(col) and df[col].isnull().all()]
cols_to_drop.extend([c for c in unnamed_null_cols if c not in cols_to_drop])

df = df.drop(columns=cols_to_drop, errors='ignore')

## 2. Fix Datetime

In [27]:
# Parse Date as datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Extract new columns based on Date before modifying it
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month_name()

# Extract only the day number from it, rename the column to Day
df['Date'] = df['Date'].dt.day
df = df.rename(columns={'Date': 'Day'})

# Parse Time column to extract Hour
if 'Time' in df.columns:
    df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
    df['Hour'] = df['Time'].dt.hour
    # Drop the redundant Time column
    df = df.drop(columns=['Time'])

# Time_of_Day categorization
def categorize_time_of_day(hour):
    if pd.isna(hour):
        return np.nan
    elif 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['Time_of_Day'] = df['Hour'].apply(categorize_time_of_day)

## 3. Standardize Categorical Columns

In [ ]:
# Strip whitespace from all string columns
string_cols = df.select_dtypes(include=['object']).columns
df[string_cols] = df[string_cols].apply(lambda x: x.str.strip())

# Standardize values
for col in ['Vehicle_Type', 'Pickup_Location', 'Drop_Location']:
    if col in df.columns:
        df[col] = df[col].str.title()

if 'Booking_Status' in df.columns:
    df['Booking_Status'] = df['Booking_Status'].str.title()

## 4. Remove Duplicates

In [ ]:
df = df.drop_duplicates()

## 5. Validate Numeric Columns

In [ ]:
for col in ['Ride_Distance', 'Booking_Value']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        negative_mask = df[col] < 0
        if negative_mask.any():
            print(f"Anomaly found: {negative_mask.sum()} negative values in {col}")
        else:
            print(f"No negative values found in {col}")

## 6. Add Useful Engineered Columns

In [ ]:
df['Is_Peak_Hour'] = df['Hour'].isin([7, 8, 9, 10, 17, 18, 19, 20, 21])
df['Is_Weekend'] = df['Day_of_Week'].isin(['Saturday', 'Sunday'])

if 'Booking_Value' in df.columns and 'Ride_Distance' in df.columns:
    # Handle divide by zero for cancelled rides
    df['Booking_Value_per_KM'] = np.where(df['Ride_Distance'] > 0, df['Booking_Value'] / df['Ride_Distance'], 0)

## 7. Handle Missing Values (Context-Aware)

In [ ]:
is_success = df['Booking_Status'] == 'Success'

# --- Successful rides: fill ratings with median ---
for col in ['Driver_Ratings', 'Customer_Rating']:
    if col in df.columns:
        median_val = df.loc[is_success, col].median()
        df.loc[is_success, col] = df.loc[is_success, col].fillna(median_val)

# --- Successful rides: drop rows where V_TAT / C_TAT / Payment_Method is null ---
cols_required_for_success = [c for c in ['V_TAT', 'C_TAT', 'Payment_Method'] if c in df.columns]
drop_mask = is_success & df[cols_required_for_success].isnull().any(axis=1)
df = df[~drop_mask]
# Refresh mask after dropping rows
is_success = df['Booking_Status'] == 'Success'

# --- Cancelled rides: fill cancellation & incomplete columns ---
for col in ['Canceled_Rides_by_Customer', 'Canceled_Rides_by_Driver', 'Incomplete_Rides', 'Incomplete_Rides_Reason']:
    if col in df.columns:
        df.loc[~is_success, col] = df.loc[~is_success, col].fillna('No_Reason')

# --- Successful rides: fill cancellation columns with Not_Applicable ---
for col in ['Canceled_Rides_by_Customer', 'Canceled_Rides_by_Driver', 'Incomplete_Rides_Reason']:
    if col in df.columns:
        df.loc[is_success, col] = df.loc[is_success, col].fillna('Not_Applicable')

# --- Cancelled rides: fill NaN in columns that only apply to completed rides ---
for col in ['V_TAT', 'C_TAT']:
    if col in df.columns:
        df.loc[~is_success, col] = df.loc[~is_success, col].fillna(0)

if 'Payment_Method' in df.columns:
    df['Payment_Method'] = df['Payment_Method'].fillna('Not_Applicable')

if 'Incomplete_Rides' in df.columns:
    df['Incomplete_Rides'] = df['Incomplete_Rides'].fillna(0)

# --- Cancelled rides: fill Driver_Ratings and Customer_Rating with 0 ---
for col in ['Driver_Ratings', 'Customer_Rating']:
    if col in df.columns:
        df.loc[~is_success, col] = df.loc[~is_success, col].fillna(0)

print(f'Shape after handling missing values: {df.shape}')
print(f'Remaining nulls:\n{df.isnull().sum()}')

## 8. Remove Outliers (IQR on Successful Rides Only)

In [ ]:
# Only remove outliers from successful rides to preserve all cancelled records
is_success = df['Booking_Status'] == 'Success'
success_part = df[is_success].copy()
cancelled_part = df[~is_success].copy()

outlier_cols = ['V_TAT', 'C_TAT', 'Ride_Distance', 'Driver_Ratings', 'Customer_Rating']
for col in outlier_cols:
    if col in success_part.columns:
        success_part[col] = pd.to_numeric(success_part[col], errors='coerce')
        Q1 = success_part[col].quantile(0.25)
        Q3 = success_part[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        before = len(success_part)
        success_part = success_part[(success_part[col] >= lower_bound) & (success_part[col] <= upper_bound)]
        print(f'{col}: removed {before - len(success_part)} outliers')

# Fill any remaining rating nulls in success after outlier removal
for col in ['Driver_Ratings', 'Customer_Rating']:
    if col in success_part.columns:
        success_part[col] = success_part[col].fillna(success_part[col].mean())

# Recombine into single dataframe
df = pd.concat([success_part, cancelled_part], ignore_index=True)
print(f'\nFinal shape after outlier removal: {df.shape}')

## 9. Export Single Cleaned Dataset

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/cleaned_dataset.csv', index=False)
print(f'Exported cleaned_dataset.csv with {len(df)} rows and {len(df.columns)} columns')

## 10. Final Verification

In [ ]:
print('=== Merged Cleaned Dataset ===')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nNull Counts:\n{df.isnull().sum()}')

print('\n=== Booking Status Distribution ===')
print(df['Booking_Status'].value_counts())

print('\n=== Value Counts ===')
for col in ['Vehicle_Type', 'Payment_Method', 'Time_of_Day']:
    if col in df.columns:
        print(f'\n{col}:')
        print(df[col].value_counts(dropna=False))

print('\n=== Sample Rows ===')
df.sample(5)